In [13]:
# Diagnosztika

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import json

# Webdriver inicializálása
driver = webdriver.Chrome()  # vagy Firefox, Edge stb.

try:
    # Oldal megnyitása
    url = "https://www.whoscored.com/matches/1914026/live/spain-laliga-2025-2026-real-betis-barcelona"
    driver.get(url)
    
    wait = WebDriverWait(driver, 20)
    
    # 1. Lépés: Cookie elfogadás gomb keresése és kattintás
    try:
        print("Cookie gomb keresése...")
        
        # Várjunk egy kicsit a cookie ablak megjelenésére
        time.sleep(2)
        
        # Próbáljuk megkeresni a gombot többféle módon
        cookie_button = None
        
        # 1. módszer: CSS selector a class és szöveg alapján
        try:
            cookie_button = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'button.Button__StyledButton-buoy__sc-a1qza5-0.elJono')
            ))
        except:
            # 2. módszer: Szöveg alapján (böngésző nyelvétől függően)
            try:
                cookie_button = wait.until(EC.element_to_be_clickable(
                    (By.XPATH, "//button[contains(text(), 'Az összes elfogadása')]")
                ))
            except:
                # 3. módszer: Button text részleges egyezés
                try:
                    cookie_button = wait.until(EC.element_to_be_clickable(
                        (By.XPATH, "//button[contains(., 'elfogadása')]")
                    ))
                except:
                    # 4. módszer: Background color alapján (ha nem változik)
                    try:
                        cookie_button = wait.until(EC.element_to_be_clickable(
                            (By.CSS_SELECTOR, 'button[style*="background-color: rgb(87, 220, 28)"]')
                        ))
                    except:
                        print("Nem található cookie gomb, lehet, hogy nem jelenik meg vagy már eltűnt")
        
        if cookie_button:
            print("Cookie gomb megtalálva, kattintás...")
            # Görgetés az elemhez (ha szükséges)
            driver.execute_script("arguments[0].scrollIntoView(true);", cookie_button)
            
            # Kattintás
            cookie_button.click()
            print("Cookie elfogadva!")
            
            # Várjunk egy kicsit, hogy bezáruljon a cookie ablak
            time.sleep(1)
        
    except Exception as cookie_error:
        print(f"Hiba a cookie gomb kezelése közben: {cookie_error}")
        print("Folytatás a Chalkboard kereséssel...")
    
    # 2. Lépés: Chalkboard link keresése és kattintás
    try:
        print("Chalkboard link keresése...")
        
        # Várakozás a Chalkboard link megjelenésére
        chalkboard_link = None
        
        try:
            # 1. módszer: CSS selector az href attribútum alapján
            chalkboard_link = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'a[href="#chalkboard"]')
            ))
        except TimeoutException:
            try:
                # 2. módszer: Link szövege alapján
                chalkboard_link = wait.until(EC.element_to_be_clickable(
                    (By.LINK_TEXT, "Chalkboard")
                ))
            except TimeoutException:
                try:
                    # 3. módszer: XPath a span szövege alapján
                    chalkboard_link = wait.until(EC.element_to_be_clickable(
                        (By.XPATH, '//a[@href="#chalkboard"]//span[contains(text(), "Chalkboard")]')
                    ))
                except TimeoutException:
                    # 4. módszer: Általánosabb keresés
                    chalkboard_link = wait.until(EC.element_to_be_clickable(
                        (By.XPATH, '//a[contains(@href, "chalkboard") or contains(.//text(), "Chalkboard")]')
                    ))
        
        if chalkboard_link:
            print("Chalkboard link megtalálva, kattintás...")
            
            # Görgetés az elemhez
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", chalkboard_link)
            
            # Várakozás, hogy az elem tényleg kattintható legyen
            time.sleep(0.5)
            
            # Kattintás
            chalkboard_link.click()
            
            print("Sikeresen rákattintottam a Chalkboard linkre!")
            
            # Várjunk egy kicsit, hogy betöltődjön a tartalom
            time.sleep(2)
            
            # Ellenőrizzük, hogy a kattintás ténylegesen történt-e
            current_url = driver.current_url
            if "#chalkboard" in current_url:
                print("Sikeresen átváltott a Chalkboard fülre!")
            else:
                print("Megjegyzés: Az URL nem tartalmazza a '#chalkboard'-ot, de a kattintás megtörtént")

            # A chalkboard megnyitása után...
            print("\n" + "="*60)
            print("CHALKBOARD ELEMZÉS")
            print("="*60)

            # 1. Nézzük meg a canvas elemeket
            print("\n=== Canvas elemek vizsgálata ===")
            try:
                canvas_elements = driver.find_elements(By.TAG_NAME, "canvas")
                for i, canvas in enumerate(canvas_elements):
                    print(f"Canvas #{i+1}:")
                    print(f"  ID: {canvas.get_attribute('id')}")
                    print(f"  Class: {canvas.get_attribute('class')}")
                    print(f"  Size: {canvas.size}")
                    print(f"  Style: {canvas.get_attribute('style')}")
                    print("---")
            except Exception as e:
                print(f"Hiba a canvas elemek vizsgálatakor: {e}")

            # 2. JavaScript segítségével nézzük meg a data attribútumokat
            print("\n=== Data attribútumok keresése (JavaScript-tel) ===")
            try:
                data_elements = driver.execute_script("""
                    var elements = document.querySelectorAll('*');
                    var result = [];
                    for (var i = 0; i < elements.length && result.length < 15; i++) {
                        var elem = elements[i];
                        var attrs = elem.attributes;
                        var dataAttrs = {};
                        for (var j = 0; j < attrs.length; j++) {
                            var attr = attrs[j];
                            if (attr.name.startsWith('data-')) {
                                dataAttrs[attr.name] = attr.value;
                            }
                        }
                        if (Object.keys(dataAttrs).length > 0) {
                            result.push({
                                tag: elem.tagName,
                                id: elem.id,
                                class: elem.className,
                                data: dataAttrs
                            });
                        }
                    }
                    return result;
                """)
                
                for elem in data_elements:
                    print(f"\n{elem['tag']}")
                    if elem['id']:
                        print(f"  ID: {elem['id']}")
                    if elem['class']:
                        print(f"  Class: {elem['class']}")
                    for key, value in elem['data'].items():
                        print(f"  {key}: {value}")
            except Exception as e:
                print(f"Hiba a data attribútumok keresésekor: {e}")

            # 3. Vizsgáljuk meg a JavaScript változókat
            print("\n=== JavaScript globális változók ===")
            js_vars_to_check = [
                "window.eventData",
                "window.matchEvents", 
                "window.chalkboard",
                "window.canvasData",
                "window.whoscored",
                "eventsData",
                "matchData",
                "window.matchCentreData",
                "window.events"
            ]

            for var_name in js_vars_to_check:
                try:
                    value = driver.execute_script(f"return {var_name} || 'nincs definiálva';")
                    print(f"{var_name}: {type(value)}")
                    if value != 'nincs definiálva':
                        if isinstance(value, (dict, list)):
                            print(f"  Tartalom részletek: {len(value) if isinstance(value, list) else len(value.keys())} elem")
                            # Csak az első néhány elemet írjuk ki
                            if isinstance(value, list):
                                print(f"  Első 3 elem: {value[:3]}")
                            else:
                                items = list(value.items())[:3]
                                print(f"  Első 3 kulcs-érték pár: {items}")
                        else:
                            print(f"  Érték: {value}")
                except Exception as e:
                    print(f"{var_name}: Hiba - {e}")

            # 4. Nézzük meg az event listener-eket a canvas-on
            print("\n=== Event listeners a Canvas-on ===")
            try:
                listeners = driver.execute_script("""
                    const canvas = document.getElementById('undefinedStatsCanvas');
                    if(!canvas) return 'Canvas nem található';
                    
                    // Próbáljuk megkapni az event listener-eket
                    const listeners = [];
                    const events = ['click', 'mousemove', 'mouseover', 'mouseout'];
                    
                    // Ez csak akkor működik, ha a getEventListeners függvény elérhető (Chrome DevTools)
                    if(typeof getEventListeners === 'function') {
                        events.forEach(eventType => {
                            const listenerList = getEventListeners(canvas)[eventType];
                            if(listenerList && listenerList.length > 0) {
                                listeners.push({
                                    event: eventType,
                                    count: listenerList.length,
                                    listeners: listenerList.map(l => l.listener.toString().substring(0, 100) + '...')
                                });
                            }
                        });
                        return listeners;
                    } else {
                        return 'getEventListeners függvény nem elérhető';
                    }
                """)
                print(f"Canvas event listeners: {listeners}")
            except Exception as e:
                print(f"Hiba az event listeners listázásakor: {e}")

            # 5. DOM struktúra vizsgálata a canvas körül
            print("\n=== DOM struktúra vizsgálata ===")
            try:
                canvas_structure = driver.execute_script("""
                    const canvas = document.getElementById('undefinedStatsCanvas');
                    if(!canvas) return null;
                    
                    // Szülő elemek felfelé haladva
                    const parents = [];
                    let current = canvas;
                    let depth = 0;
                    
                    while(current && depth < 6) {
                        const children = [];
                        for(let i = 0; i < Math.min(current.children.length, 5); i++) {
                            const child = current.children[i];
                            children.push({
                                tag: child.tagName,
                                id: child.id,
                                className: child.className
                            });
                        }
                        
                        parents.push({
                            depth: depth,
                            tag: current.tagName,
                            id: current.id,
                            className: current.className,
                            childrenCount: current.children.length,
                            childrenSample: children
                        });
                        current = current.parentElement;
                        depth++;
                    }
                    
                    return parents;
                """)
                
                if canvas_structure:
                    for parent in canvas_structure:
                        indent = "  " * parent['depth']
                        print(f"{indent}{parent['tag']}", end="")
                        if parent['id']:
                            print(f"#{parent['id']}", end="")
                        if parent['className']:
                            print(f".{parent['className'][:50]}", end="")
                        print(f" ({parent['childrenCount']} gyermek)")
                else:
                    print("Canvas nem található")
            except Exception as e:
                print(f"Hiba a DOM struktúra vizsgálatakor: {e}")

            # 6. Esemény típusok tesztelése
            print("\n=== Esemény típusok tesztelése ===")
            try:
                event_filters = driver.find_elements(By.CSS_SELECTOR, "#event-type-filters li")
                print(f"Összesen {len(event_filters)} eseménytípus található")
                
                # Kattintsunk a "Shots"-ra (0. elem)
                if len(event_filters) > 0:
                    shots_filter = event_filters[0]
                    event_name = shots_filter.find_element(By.TAG_NAME, "h4").text
                    print(f"\nKattintás a '{event_name}'-ra...")
                    
                    # Kattintás előtt
                    before_state = driver.execute_script("""
                        const canvas = document.getElementById('undefinedStatsCanvas');
                        return {
                            visible: canvas && canvas.offsetWidth > 0 && canvas.offsetHeight > 0,
                            hasEvents: document.querySelectorAll('.player[data-has-events="true"]').length
                        };
                    """)
                    print(f"Kattintás előtt: {before_state}")
                    
                    # Kattintás
                    shots_filter.click()
                    time.sleep(1.5)
                    
                    # Nézzük meg, változott-e valami
                    after_state = driver.execute_script("""
                        // Nézzük meg, van-e látható elem a canvas-en
                        const canvas = document.getElementById('undefinedStatsCanvas');
                        let hasVisibleElements = false;
                        
                        if(canvas) {
                            const ctx = canvas.getContext('2d');
                            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
                            for(let i = 3; i < imageData.data.length; i += 4) {
                                if(imageData.data[i] > 10) {
                                    hasVisibleElements = true;
                                    break;
                                }
                            }
                        }
                        
                        return {
                            hasVisibleElements: hasVisibleElements,
                            playerEventCount: document.querySelectorAll('.player[data-has-events="true"]').length,
                            selectedStats: document.querySelectorAll('.selected-stat-value[data-value]').length
                        };
                    """)
                    print(f"Kattintás után: {after_state}")
                    
                    # Most nézzük meg, vannak-e SVG elemek vagy más grafikus elemek
                    svg_elements = driver.execute_script("""
                        return {
                            svgCount: document.querySelectorAll('svg').length,
                            imgCount: document.querySelectorAll('img').length,
                            divCount: document.querySelectorAll('div').length,
                            canvasCount: document.querySelectorAll('canvas').length
                        };
                    """)
                    print(f"Grafikus elemek: {svg_elements}")
                    
            except Exception as e:
                print(f"Hiba az eseménytípus tesztelésekor: {e}")

            # 7. Próbáljunk meg kattintani a canvas-en és tooltip-eket keresni
            print("\n=== Tooltip keresés ===")
            try:
                # Először kattintsunk a canvas egy pontjára
                canvas = driver.find_element(By.ID, "undefinedStatsCanvas")
                
                # Mozgassuk az egeret a canvas közepére
                from selenium.webdriver.common.action_chains import ActionChains
                actions = ActionChains(driver)
                actions.move_to_element_with_offset(canvas, 200, 150).perform()
                time.sleep(0.5)
                
                # Kattintsunk
                actions.click().perform()
                time.sleep(1)
                
                # Nézzük meg, megjelent-e tooltip
                tooltips = driver.execute_script("""
                    const tooltips = document.querySelectorAll('[class*="tooltip"], [class*="info"], #player-event-details');
                    return Array.from(tooltips).map(t => ({
                        html: t.outerHTML.substring(0, 200),
                        text: t.innerText,
                        visible: t.offsetWidth > 0 && t.offsetHeight > 0
                    }));
                """)
                
                if tooltips:
                    print(f"Tooltip-ek találva: {len(tooltips)}")
                    for i, tip in enumerate(tooltips):
                        print(f"  Tooltip #{i+1}:")
                        print(f"    Látható: {tip['visible']}")
                        print(f"    Szöveg: {tip['text'][:100]}...")
                else:
                    print("Nem található tooltip")
                    
            except Exception as e:
                print(f"Hiba a tooltip keresésekor: {e}")

            # 8. Nézzük meg a hálózati forgalmat (performance API)
            print("\n=== Hálózati kérések (Performance API) ===")
            try:
                perf_data = driver.execute_script("""
                    const resources = performance.getEntriesByType('resource');
                    const xhrRequests = resources.filter(r => 
                        r.initiatorType === 'xmlhttprequest' || 
                        r.name.includes('event') ||
                        r.name.includes('match') ||
                        r.name.includes('data')
                    );
                    
                    return xhrRequests.map(r => ({
                        name: r.name,
                        type: r.initiatorType,
                        duration: Math.round(r.duration),
                        size: r.transferSize ? Math.round(r.transferSize / 1024) + ' KB' : 'ismeretlen'
                    })).slice(0, 10);
                """)
                
                if perf_data:
                    print(f"Hálózati kérések találva: {len(perf_data)}")
                    for req in perf_data:
                        print(f"  - {req['name'][:80]}... (típus: {req['type']}, idő: {req['duration']}ms, méret: {req['size']})")
                else:
                    print("Nem található releváns hálózati kérés")
            except Exception as e:
                print(f"Hiba a hálózati kérések lekérdezésekor: {e}")

            # 9. Próbáljuk meg kinyerni az események adatait a játékosokról
            print("\n=== Játékos események adatainak kinyerése ===")
            try:
                player_events = driver.execute_script("""
                    const players = document.querySelectorAll('.player');
                    const result = [];
                    
                    for(const player of players) {
                        const playerId = player.getAttribute('data-player-id');
                        const statValue = player.querySelector('.selected-stat-value');
                        const hasEvents = player.hasAttribute('data-has-events');
                        
                        if(hasEvents && statValue && statValue.getAttribute('data-value')) {
                            result.push({
                                id: playerId,
                                name: player.querySelector('.player-name')?.title || player.querySelector('.player-name')?.textContent,
                                statValue: statValue.getAttribute('data-value'),
                                position: player.querySelector('.position')?.textContent,
                                shirtNumber: player.querySelector('.shirt-number')?.textContent
                            });
                        }
                    }
                    
                    return result;
                """)
                
                if player_events:
                    print(f"Játékosok eseményekkel: {len(player_events)}")
                    for player in player_events[:5]:  # Csak az első 5
                        print(f"  - {player['shirtNumber']}. {player['name']} ({player['position']}): {player['statValue']} esemény")
                else:
                    print("Nem található játékos eseményadatok")
            except Exception as e:
                print(f"Hiba a játékos események kinyerésekor: {e}")

            print("\n" + "="*60)
            print("ELEMZÉS VÉGE")
            print("="*60)
                
    except Exception as chalkboard_error:
        print(f"Hiba a Chalkboard link kezelése közben: {chalkboard_error}")

except Exception as e:
    print(f"Általános hiba történt: {str(e)}")
    import traceback
    traceback.print_exc()

finally:
    # Opcionális: várakozás, hogy láthasd az eredményt
    print("\n" + "="*50)
    sleep_time = 3
    print(f"A böngésző nyitva marad {sleep_time} másodpercig...")
    print("="*50 + "\n")
    
    time.sleep(sleep_time)
    driver.quit()

Cookie gomb keresése...
Cookie gomb megtalálva, kattintás...
Cookie elfogadva!
Chalkboard link keresése...
Chalkboard link megtalálva, kattintás...
Sikeresen rákattintottam a Chalkboard linkre!
Megjegyzés: Az URL nem tartalmazza a '#chalkboard'-ot, de a kattintás megtörtént

CHALKBOARD ELEMZÉS

=== Canvas elemek vizsgálata ===
Canvas #1:
  ID: undefinedPitchCanvas
  Class: 
  Size: {'height': 534, 'width': 914}
  Style: position: absolute;
---
Canvas #2:
  ID: undefinedStatsCanvas
  Class: 
  Size: {'height': 533, 'width': 908}
  Style: margin-top: 2px; margin-left: 2px; position: absolute;
---

=== Data attribútumok keresése (JavaScript-tel) ===

SCRIPT
  ID: adagiojs-2b8150e6f9e49c
  data-pid: 1205

STYLE
  data-styled: active
  data-styled-version: 5.3.6

STYLE
  data-styled: active
  data-styled-version: 5.3.6

SCRIPT
  data-kleanads: true

SCRIPT
  data-kleanads: true

SCRIPT
  data-kleanads: true

SCRIPT
  data-kleanads: true

SCRIPT
  data-kleanads: true

SCRIPT
  data-kleanads:

In [14]:
# Adatok kinyerése

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json

# Webdriver inicializálása
driver = webdriver.Chrome()

try:
    # Oldal megnyitása (a korábbi kódod alapján)
    url = "https://www.whoscored.com/matches/1914026/live/spain-laliga-2025-2026-real-betis-barcelona"
    driver.get(url)
    
    wait = WebDriverWait(driver, 20)
    
    # Cookie elfogadás (egyszerűsítve)
    time.sleep(2)
    try:
        cookie_btn = driver.find_element(By.XPATH, "//button[contains(., 'elfogadása')]")
        cookie_btn.click()
        time.sleep(1)
    except:
        pass
    
    # Chalkboard megnyitása
    try:
        chalkboard_link = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a[href="#chalkboard"]')))
        chalkboard_link.click()
        time.sleep(2)
    except:
        # Alternatív keresés
        chalkboard_link = driver.find_element(By.XPATH, "//a[contains(@href, 'chalkboard')]")
        chalkboard_link.click()
        time.sleep(2)
    
    print("Chalkboard megnyitva!")
    
    # 1. PRÓBÁLJUK MEG KINYERNI AZ ESEMÉNYADATOKAT
    
    print("\n=== 1. JavaScript változók tartalmának kinyerése ===")
    
    # Próbáljuk megkapni a JSON adatokat
    event_data_sources = [
        "window.eventData",
        "window.matchEvents",
        "window.matchCentreData",
        "window.events",
        "window.matchCentreData ? window.matchCentreData.events : null",
        "window.matchCentreData ? window.matchCentreData.matchCentreData : null",
        "window.WSM ? window.WSM.matchData : null",
        "window.whoscored ? JSON.parse(window.whoscored) : null"
    ]
    
    found_data = None
    for source in event_data_sources:
        try:
            data = driver.execute_script(f"return {source}")
            if data and data != 'nincs definiálva' and data != 'null':
                print(f"\nTalált adat a következő forrásban: {source}")
                
                # Ha string, próbáljuk JSON-ná alakítani
                if isinstance(data, str):
                    try:
                        parsed = json.loads(data)
                        print(f"  String -> JSON sikerült, {len(parsed) if isinstance(parsed, list) else len(parsed.keys())} elem")
                        found_data = parsed
                        break
                    except json.JSONDecodeError:
                        print(f"  String, de nem JSON: első 200 karakter: {data[:200]}...")
                        found_data = data
                        break
                else:
                    print(f"  Típus: {type(data)}, méret: {len(data) if isinstance(data, (list, dict)) else 'N/A'}")
                    found_data = data
                    break
        except Exception as e:
            print(f"  {source}: {e}")
            continue
    
    # 2. HA TALÁLTUNK ADATOT, NÉZZÜK MEG A TARTALMÁT
    
    if found_data:
        print(f"\n=== 2. Talált adatok elemzése ===")
        
        # Ha dictionary
        if isinstance(found_data, dict):
            print("Dictionary találva, kulcsok:")
            for key in found_data.keys():
                value = found_data[key]
                print(f"  '{key}': {type(value)}", end="")
                if isinstance(value, (list, dict)):
                    print(f" ({len(value)} elem)")
                else:
                    print(f" - {str(value)[:100]}...")
            
            # Keresünk events kulcsot
            if 'events' in found_data:
                print(f"\n'events' kulcs találva: {len(found_data['events'])} esemény")
                # Nézzük meg az első néhány eseményt
                for i, event in enumerate(found_data['events'][:3]):
                    print(f"  Esemény {i+1}: {event}")
            
            # Keresünk matchCentreData-t
            if 'matchCentreData' in found_data:
                print(f"\n'matchCentreData' találva")
                match_data = found_data['matchCentreData']
                if isinstance(match_data, dict):
                    for key in match_data.keys():
                        print(f"  '{key}': {type(match_data[key])}")
        
        # Ha lista
        elif isinstance(found_data, list):
            print(f"Lista találva: {len(found_data)} elem")
            # Nézzük meg az első néhány elemet
            for i, item in enumerate(found_data[:3]):
                print(f"  Elem {i+1}: típus: {type(item)}")
                if isinstance(item, dict):
                    print(f"    Kulcsok: {list(item.keys())[:5]}...")
        
        # Ha string
        elif isinstance(found_data, str):
            print(f"String találva, hossz: {len(found_data)} karakter")
            print(f"Első 500 karakter: {found_data[:500]}...")
    
    # 3. HA NEM TALÁLTUNK, PRÓBÁLJUNK MÁSKÉPP
    
    if not found_data:
        print("\n=== 3. Alternatív keresési módszerek ===")
        
        # Próbáljuk meg a localStorage-t
        try:
            print("\nLocalStorage adatok keresése...")
            localStorage_data = driver.execute_script("""
                const data = {};
                for(let i = 0; i < localStorage.length; i++) {
                    const key = localStorage.key(i);
                    if(key.includes('event') || key.includes('match') || key.includes('chalkboard')) {
                        try {
                            data[key] = JSON.parse(localStorage.getItem(key));
                        } catch(e) {
                            data[key] = localStorage.getItem(key);
                        }
                    }
                }
                return data;
            """)
            
            if localStorage_data:
                print(f"LocalStorage adatok találva: {len(localStorage_data)} kulcs")
                for key, value in localStorage_data.items():
                    print(f"  '{key}': {type(value)}")
                    if isinstance(value, dict):
                        print(f"    Kulcsok: {list(value.keys())[:5]}...")
                    elif isinstance(value, str):
                        print(f"    Előzetes: {value[:100]}...")
        except Exception as e:
            print(f"LocalStorage hiba: {e}")
        
        # Próbáljuk meg a script tag-eket
        try:
            print("\nScript tag-ek keresése event adatokkal...")
            scripts = driver.execute_script("""
                const scripts = document.querySelectorAll('script');
                const results = [];
                for(const script of scripts) {
                    const text = script.textContent || script.innerText;
                    if(text.includes('event') && (text.includes('x') || text.includes('y') || text.includes('coordinate'))) {
                        results.push(text.substring(0, 500));
                    }
                }
                return results;
            """)
            
            if scripts:
                print(f"{len(scripts)} script találva eseményadatokat tartalmazhat")
                for i, script in enumerate(scripts[:2]):
                    print(f"  Script {i+1}: {script}...")
        except Exception as e:
            print(f"Script keresés hiba: {e}")
    
    # 4. PRÓBÁLJUK MEG KAPNI A TELJES MATCH CENTRE DATA-T
    
    print("\n=== 4. Teljes match centre adatok kinyerése ===")
    
    try:
        # Ez a legvalószínűbb, hogy itt vannak az adatok
        all_data = driver.execute_script("""
            // Próbáljuk meg összegyűjteni minden lehetséges adatot
            const data = {};
            
            // 1. Window változók
            const windowVars = ['eventData', 'matchEvents', 'matchCentreData', 'events', 'whoscored', 'WSM'];
            windowVars.forEach(key => {
                if(window[key]) {
                    try {
                        if(typeof window[key] === 'string') {
                            data[key] = JSON.parse(window[key]);
                        } else {
                            data[key] = window[key];
                        }
                    } catch(e) {
                        data[key] = window[key];
                    }
                }
            });
            
            // 2. Nézzük meg, van-e init esemény
            data['hasEventListeners'] = false;
            const canvas = document.getElementById('undefinedStatsCanvas');
            if(canvas) {
                data['canvas'] = {
                    id: canvas.id,
                    width: canvas.width,
                    height: canvas.height
                };
            }
            
            // 3. Játékos események száma
            const playersWithEvents = document.querySelectorAll('.player[data-has-events="true"]');
            data['playersWithEvents'] = playersWithEvents.length;
            
            return data;
        """)
        
        print(f"Összegyűjtött adatok:")
        for key, value in all_data.items():
            if key == 'canvas':
                print(f"  {key}: {value}")
            elif key == 'playersWithEvents':
                print(f"  {key}: {value}")
            else:
                print(f"  {key}: {type(value)}")
                if value and isinstance(value, dict):
                    print(f"    Kulcsok: {list(value.keys())[:10]}...")
                elif value and isinstance(value, list):
                    print(f"    Elemek száma: {len(value)}")
    
    except Exception as e:
        print(f"Hiba az adatok összegyűjtésekor: {e}")
    
    # 5. ESEMÉNYTÍPUSOK VÉGIGLÉPÉSE ÉS ADATOK GYŰJTÉSE
    
    print("\n=== 5. Eseménytípusok végiglépése ===")
    
    try:
        event_filters = driver.find_elements(By.CSS_SELECTOR, "#event-type-filters li")
        event_data_by_type = {}
        
        for i in range(min(3, len(event_filters))):  # Csak az első 3 típust
            try:
                # Vissza a kezdőállapotba
                if i > 0:
                    event_filters[0].click()
                    time.sleep(1)
                
                # Kattintás az aktuális típusra
                current_filter = event_filters[i]
                event_name = current_filter.find_element(By.TAG_NAME, "h4").text
                print(f"\n'{event_name}' eseménytípus...")
                current_filter.click()
                time.sleep(1.5)
                
                # Adatok kinyerése
                event_data = driver.execute_script("""
                    // Próbáljuk megkapni az eseményeket
                    const eventElements = document.querySelectorAll('[data-event-type], .event-marker, [class*="event-"]');
                    const events = [];
                    
                    eventElements.forEach(el => {
                        const rect = el.getBoundingClientRect();
                        const style = window.getComputedStyle(el);
                        events.push({
                            x: rect.x + rect.width/2,
                            y: rect.y + rect.height/2,
                            width: rect.width,
                            height: rect.height,
                            backgroundColor: style.backgroundColor,
                            display: style.display,
                            html: el.outerHTML.substring(0, 200)
                        });
                    });
                    
                    // Canvas pixel adatok
                    const canvas = document.getElementById('undefinedStatsCanvas');
                    let canvasData = null;
                    if(canvas) {
                        const ctx = canvas.getContext('2d');
                        // Csak egy kis területet nézünk
                        const imageData = ctx.getImageData(0, 0, 50, 50);
                        canvasData = {
                            width: 50,
                            height: 50,
                            dataLength: imageData.data.length,
                            hasContent: Array.from(imageData.data).some(v => v > 0)
                        };
                    }
                    
                    return {
                        eventElements: events,
                        canvasData: canvasData,
                        eventElementCount: events.length
                    };
                """)
                
                event_data_by_type[event_name] = event_data
                print(f"  {event_data['eventElementCount']} esemény elem találva")
                print(f"  Canvas adatok: {event_data['canvasData']}")
                
                # Nézzük meg az első pár elemet
                for j, elem in enumerate(event_data['eventElements'][:2]):
                    print(f"    Elem {j+1}: ({elem['x']}, {elem['y']}), szín: {elem['backgroundColor']}")
                
            except Exception as e:
                print(f"  Hiba: {e}")
                continue
        
        print(f"\nÖsszegyűjtött {len(event_data_by_type)} eseménytípus adata")
        
    except Exception as e:
        print(f"Hiba az eseménytípusok végiglépésekor: {e}")
    
    # 6. VÉGLEGES KISERLET - PRÓBÁLJUK MEGKAPNI A RAW ADATOKAT
    
    print("\n=== 6. Raw adatok keresése ===")
    
    # Próbáljuk meg minden lehetséges módon
    final_attempts = [
        # Leggyakoribb WhoScored adatstruktúrák
        "return (window.WSM || {}).data || (window.WSM || {}).matchData;",
        "return window._sharedData || window.__NUXT__ || window.__NEXT_DATA__;",
        "return window.initialState || window.initialData || window.preloadedState;",
        # Network response-ok (ha vannak)
        "return window.performance.getEntriesByType('resource').filter(r => r.name.includes('match') || r.name.includes('event')).map(r => r.name);",
        # Session storage
        "const data = {}; for(let i = 0; i < sessionStorage.length; i++) {const k = sessionStorage.key(i); if(k.includes('match') || k.includes('event')) {data[k] = sessionStorage.getItem(k);}} return data;"
    ]
    
    for attempt in final_attempts:
        try:
            result = driver.execute_script(attempt)
            if result and (isinstance(result, (dict, list)) or (isinstance(result, str) and len(result) > 100)):
                print(f"\nSikeres adatgyűjtés: {attempt[:50]}...")
                print(f"  Eredmény típusa: {type(result)}")
                if isinstance(result, dict):
                    print(f"  Kulcsok: {list(result.keys())[:10]}")
                elif isinstance(result, list):
                    print(f"  Elemek száma: {len(result)}")
                    if result and isinstance(result[0], dict):
                        print(f"  Első elem kulcsai: {list(result[0].keys())[:5]}")
                break
        except:
            continue
    
    print("\n" + "="*60)
    print("ADATG YŰJTÉS BEFEJEZVE")
    print("="*60)

except Exception as e:
    print(f"Általános hiba: {e}")
    import traceback
    traceback.print_exc()

finally:
    print("\nVárakozás a böngésző bezárása előtt...")
    time.sleep(5)
    driver.quit()

Chalkboard megnyitva!

=== 1. JavaScript változók tartalmának kinyerése ===

=== 3. Alternatív keresési módszerek ===

LocalStorage adatok keresése...

Script tag-ek keresése event adatokkal...
12 script találva eseményadatokat tartalmazhat
  Script 1: (function (w, d, s, l, i) {
            w[l] = w[l] || []; w[l].push({
                'gtm.start':

                    new Date().getTime(), event: 'gtm.js'
            }); var f = d.getElementsByTagName(s)[0],

                j = d.createElement(s), dl = l != 'dataLayer' ? '&l=' + l : ''; j.async = true; j.src =

                    'https://www.googletagmanager.com/gtm.js?id=' + i + dl; f.parentNode.insertBefore(j, f);

        })(window, document, 'script', 'dataLayer', 'GTM-K2NSL35');...
  Script 2: (function(){ var kleanads=function(e){"use strict";const t="adm";const i="kleanads_errors";class n{constructor(e,t,i,n=1){this.tagId=e,this.kleanadsVersion=t,this.throttling=n,this.device=i||"na"}handleMessage(e,t,i){const n={source:t?